# Training a Potential: how to setup data pipeline, model and trainer 

# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [1]:
%load_ext autoreload
%autoreload 2

import logging
molpot_logger = logging.getLogger('molpot')
molpot_logger.setLevel(logging.DEBUG)
import molpot as mpot
import torch
from molpot import alias

In [4]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cpu",
    preprocess=[
        mpot.locality.NeighborList(
            cutoff=5,
        )
    ],
)

data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 1000

                           dataset: md17                           
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ label  ┃          type          ┃    unit    ┃     comment      ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ energy │    <class 'float'>     │  kcal/mol  │ potential_energy │
│ forces │ <class 'torch.Tensor'> │ kcal/mol/A │      forces      │
└────────┴────────────────────────┴────────────┴──────────────────┘

In [16]:
train_ds, valid_ds = torch.utils.data.random_split(rmd17_ds, [.8, .2])

train_dl = mpot.DataLoader(
    dataset=train_ds,
    batch_size=10,
    shuffle=False,
)
valid_dl = mpot.DataLoader(
    dataset=valid_ds,
    batch_size=10,
    shuffle=False,
)

In [17]:
for batch in train_dl:
    print(batch)
    break

TensorDict(
    fields={
        angles: TensorDict(
            fields={
            },
            batch_size=torch.Size([10]),
            device=None,
            is_shared=False),
        atoms: TensorDict(
            fields={
                Z: Tensor(shape=torch.Size([10, 21]), device=cpu, dtype=torch.uint8, is_shared=False),
                atomic_batch_mask: Tensor(shape=torch.Size([10, 21]), device=cpu, dtype=torch.int64, is_shared=False),
                xyz: Tensor(shape=torch.Size([10, 21, 3]), device=cpu, dtype=torch.float64, is_shared=False)},
            batch_size=torch.Size([10]),
            device=None,
            is_shared=False),
        bonds: TensorDict(
            fields={
            },
            batch_size=torch.Size([10]),
            device=None,
            is_shared=False),
        box: TensorDict(
            fields={
            },
            batch_size=torch.Size([10]),
            device=None,
            is_shared=False),
        cell: TensorDi

## Setting up the model

In [ ]:
cutoff = 5.0
n_rbf = 10
arch = mpot.nnp.PiNet(
    depth=4,
    basis_fn=mpot.nnp.GaussianRBF(n_rbf, cutoff),
    cutoff_fn=mpot.nnp.CosineCutoff(cutoff),
    pp_nodes=[16],
    pi_nodes=[16],
    ii_nodes=[16],
    max_atomtypes=10,
)
energy_readout = mpot.nnp.Atomwise(16, 1, [], from_key=("pinet", "p1"), to_key=("predicts", "U"), aggregation_mode="add")

model = mpot.PotentialSeq("pinet", arch, energy_readout)

In [ ]:
import torch

loss_fn = mpot.engine.loss.loss_wrapper(
    input_=("predicts", "U"), target=("rmd17", "energy")
)(torch.nn.MSELoss())
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, 0.994)

trainer = mpot.PotentialTrainer(
    "pinet-rmd17",
    model,
    loss_fn,
    optimizer,
    scheduler,
    root_dir="runs",
)

profiler = mpot.engine.fix.tensorboard.Profiler(wait=1, warmup=1, active=3)

trainer.fix.register(profiler.before_epoch, trainer.Stage.before_epoch)
trainer.fix.register(profiler.after_epoch, trainer.Stage.after_epoch)
trainer.fix.register(profiler.after_step, trainer.Stage.after_step)

trainer.fix.register(
    mpot.engine.fix.TensorBoardFix(
        every_n_steps=1,
        log_dir="runs/pinet-rmd17",
    ),
    trainer.Stage.after_step,
)
trainer.fix.register(
    mpot.engine.fix.EpochSpeed(every_n_epochs=1), trainer.Stage.after_epoch
)
trainer.fix.register(
    mpot.engine.fix.StepSpeed(every_n_steps=100), trainer.Stage.after_step
)
trainer.fix.register(
    mpot.engine.fix.MAE(
        output_key="e_mae",
        every_n_steps=100,
        result_key=("predicts", "U"),
        target_key=("rmd17", "energy"),
    ),
    trainer.Stage.after_step,
)
# TODO: ckpt, valid
trainer.fix.register(mpot.engine.fix.Validation(1, valid_dl), trainer.Stage.after_epoch)

cuda


In [ ]:
%load_ext tensorboard
# trainer.compile()
inputs = trainer.train(train_dl, 100)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


RuntimeError: zero-dimensional tensor (at position 0) cannot be concatenated

In [ ]:
import math
class MyIterableDataset(torch.utils.data.IterableDataset):
    def __init__(self, start, end):
        super(MyIterableDataset).__init__()
        assert end > start, "this example code only works with end >= start"
        self.start = start
        self.end = end
    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:  # single-process data loading, return the full iterator
            iter_start = self.start
            iter_end = self.end
        else:  # in a worker process
            # split workload
            per_worker = int(math.ceil((self.end - self.start) / float(worker_info.num_workers)))
            worker_id = worker_info.id
            iter_start = self.start + worker_id * per_worker
            iter_end = min(iter_start + per_worker, self.end)
        return iter(range(iter_start, iter_end))
# should give same set of data as range(3, 7), i.e., [3, 4, 5, 6].
ds = MyIterableDataset(start=3, end=7)

# Single-process loading
print(list(torch.utils.data.DataLoader(ds, num_workers=0)))

# Mult-process loading with two worker processes
# Worker 0 fetched [3, 4].  Worker 1 fetched [5, 6].
print(list(torch.utils.data.DataLoader(ds, num_workers=2)))

# With even more workers
print(list(torch.utils.data.DataLoader(ds, num_workers=4)))

[tensor([3]), tensor([4]), tensor([5]), tensor([6])]
[tensor([3]), tensor([5]), tensor([4]), tensor([6])]
[tensor([3]), tensor([4]), tensor([5]), tensor([6])]


In [ ]:
for data in torch.utils.data.DataLoader(ds, num_workers=4):
    print(data)

tensor([3])
tensor([4])
tensor([5])
tensor([6])


# Training a Classic Potential

This section introduces how to use `MolPot` to train a classic forcefield

In [ ]:
import molpy as mp

ModuleNotFoundError: No module named 'molpy'

In [ ]:
ff = mp.ForceField("gaff")

atomstyle = ff.def_atomstyle("full")
C = atomstyle.def_atomtype("C", 1, {"mass": 12.01})
H = atomstyle.def_atomtype("H", 2, {"mass": 1.01})
O = atomstyle.def_atomtype("O", 3, {"mass": 16.00})
N = atomstyle.def_atomtype("N", 4, {"mass": 14.01})

bondstyle = ff.def_bondstyle("harmonic")